# Project Deliverable 4: Nonlinear Classification Model

## Section 0: Setup

- Spark session init
- Load the 3 .dat files
- Cleaning steps (keep the filters even though data was clean — defensive code)
- Join into one DataFrame
- Feature engineering (all your derived columns)
- Train/test split (seed=42)

In [1]:
# Verify Java installation
import os
print("JAVA_HOME is:", os.environ.get("JAVA_HOME"))
!java -version

JAVA_HOME is: /Library/Java/JavaVirtualMachines/temurin-17.jdk/Contents/Home
openjdk version "17.0.16" 2025-07-15
OpenJDK Runtime Environment Temurin-17.0.16+8 (build 17.0.16+8)
OpenJDK 64-Bit Server VM Temurin-17.0.16+8 (build 17.0.16+8, mixed mode, sharing)


In [2]:
import os
import sys

from pyspark.sql.functions import broadcast, explode, split, col, avg, round, desc, when, count, size, regexp_extract, length
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [3]:
import os
os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-17.jdk/Contents/Home"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = SparkSession.builder \
    .appName("Team Vortex MovieLens") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")  # Suppress warnings for cleaner output

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/18 20:01:14 WARN Utils: Your hostname, Mostafizurs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.171 instead (on interface en0)
26/04/18 20:01:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/18 20:01:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
# Create a SparkSession called 'spark'
spark = SparkSession.builder \
    .appName("Week 6 Project 1 Works")\
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")  # Suppress warnings for cleaner output

In [5]:
schema_users = StructType([
    StructField("UserID", IntegerType(), True),
    StructField("Gender", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Occupation",IntegerType(), True),
    StructField("Zip-code", StringType(), True)
])

df_users = spark.read \
    .option("header", "false") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("sep", "::") \
    .schema(schema_users) \
    .csv("./ml-1m/users.dat")

df_users.show(5)

schema_ratings  = StructType([
    StructField("UserID", IntegerType(), True),
    StructField("MovieID", IntegerType(), True),
    StructField("Rating", IntegerType(), True),
    StructField("Timestamp",IntegerType(), True)
])

df_ratings = spark.read \
    .option("header", "false") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("sep", "::") \
    .schema(schema_ratings) \
    .csv("./ml-1m/ratings.dat")

df_ratings.show(5)

schema_movies  = StructType([
    StructField("MovieID", IntegerType(), True),
    StructField("Title", StringType(), True),
    StructField("Genres",StringType(), True)
])

df_movies = spark.read \
    .option("header", "false") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("sep", "::") \
    .schema(schema_movies) \
    .csv("./ml-1m/movies.dat")

df_movies.show(5)


+------+------+---+----------+--------+
|UserID|Gender|Age|Occupation|Zip-code|
+------+------+---+----------+--------+
|     1|     F|  1|        10|   48067|
|     2|     M| 56|        16|   70072|
|     3|     M| 25|        15|   55117|
|     4|     M| 45|         7|   02460|
|     5|     M| 25|        20|   55455|
+------+------+---+----------+--------+
only showing top 5 rows
+------+-------+------+---------+
|UserID|MovieID|Rating|Timestamp|
+------+-------+------+---------+
|     1|   1193|     5|978300760|
|     1|    661|     3|978302109|
|     1|    914|     3|978301968|
|     1|   3408|     4|978300275|
|     1|   2355|     5|978824291|
+------+-------+------+---------+
only showing top 5 rows
+-------+--------------------+--------------------+
|MovieID|               Title|              Genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Animation|Childre...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men 

### Cleaning

#### Duplicate Ratings 

In [6]:
# Only run this if dup_before.count() was > 0
df_ratings = df_ratings.dropDuplicates(["UserID", "MovieID"])

#### Referential Integrity

In [7]:
df_ratings = df_ratings.join(df_users.select("UserID"), on="UserID", how="inner")
df_ratings = df_ratings.join(df_movies.select("MovieID"), on="MovieID", how="inner")

In [8]:

# 1) Join ratings with user/movie metadata first
base_df = (
    df_ratings
    .join(broadcast(df_users), on="UserID", how="inner")
    .join(broadcast(df_movies), on="MovieID", how="inner")
    .filter((col("Rating") >= 1) & (col("Rating") <= 5))
    .fillna({"Genres": "Unknown", "Title": "Unknown"})
)


In [9]:
base_df.describe().show()

[Stage 12:>                                                         (0 + 1) / 1]

+-------+------------------+------------------+------------------+--------------------+-------+------------------+-----------------+-----------------+--------------------+-------+
|summary|           MovieID|            UserID|            Rating|           Timestamp| Gender|               Age|       Occupation|         Zip-code|               Title| Genres|
+-------+------------------+------------------+------------------+--------------------+-------+------------------+-----------------+-----------------+--------------------+-------+
|  count|           1000209|           1000209|           1000209|             1000209|1000209|           1000209|          1000209|          1000209|             1000209|1000209|
|   mean|1865.5398981612843| 3024.512347919285| 3.581564453029317| 9.722436954046655E8|   NULL| 29.73831369243828|8.036138447064564|223239.8917114074|                NULL|   NULL|
| stddev|1096.0406894572475|1728.4126948999872|1.1171018453732586|1.2152558939916113E7|   NULL|11.75

In [10]:
# high_rating
base_df = base_df.withColumn("high_rating", when(col("Rating") >= 4, 1).otherwise(0))

train_df, test_df = base_df.randomSplit([0.8, 0.2], seed=42)

print("Train:", train_df.count())
print("Test:", test_df.count())


Train: 800391


[Stage 29:>                                                         (0 + 8) / 8]

Test: 199818


In [11]:

from pyspark.sql import functions as F
from pyspark.sql.functions import col, when

# 4) Compute rating-based stats from TRAIN ONLY
user_stats = train_df.groupBy("UserID").agg(
    avg("Rating").alias("user_avg_rating"),
    count("*").alias("user_rating_count")
)

movie_stats = train_df.groupBy("MovieID").agg(
    avg("Rating").alias("movie_avg_rating"),
    count("*").alias("movie_popularity")
)

# 5) Global fallback values for unseen users/movies in test/holdout
overall_rating = train_df.agg(avg("Rating").alias("overall")).first()["overall"]

def add_features(df):
    out = (
        df.join(user_stats, on="UserID", how="left")
          .join(movie_stats, on="MovieID", how="left")
          .withColumn("gender_encoded", when(col("Gender") == "F", 1).otherwise(0))
          .withColumn("num_genres", F.size(F.split(col("Genres"), "\\|")))
          .withColumn(
              "release_year",
              F.when(F.length(F.regexp_extract(col("Title"), r"\((\d{4})\)", 1)) == 4,
                     F.regexp_extract(col("Title"), r"\((\d{4})\)", 1).cast("int"))
          )
          .withColumn(
              "movie_age",
              F.when(col("release_year").isNotNull(), 2000 - col("release_year"))
          )
    )

    # Fill nulls from unseen IDs or missing metadata
    out = out.fillna({
        "user_avg_rating": overall_rating,
        "movie_avg_rating": overall_rating,
        "movie_popularity": 0,
        "user_rating_count": 0,
        "release_year": 2000,
        "movie_age": 0
    })

    # Interaction features after the stats exist
    out = out.withColumn("user_movie_diff", F.abs(col("user_avg_rating") - col("movie_avg_rating")))
    out = out.withColumn("user_movie_interaction", col("user_avg_rating") * col("movie_avg_rating"))
    out = out.withColumn(
        "popularity_bucket",
        when(col("movie_popularity") < 50, 0).when(col("movie_popularity") < 200, 1).otherwise(2)
    )
    out = out.withColumn(
        "user_activity_level",
        when(col("user_rating_count") < 20, 0).when(col("user_rating_count") < 100, 1).otherwise(2)
    )
    out = out.withColumn(
        "movie_age_bucket",
        when(col("movie_age") < 5, 0).when(col("movie_age") < 15, 1).otherwise(2)
    )
    return out

train_feat = add_features(train_df)
test_feat = add_features(test_df)



## Section 1: Model Construction

#### 1a. Model Selection

**Comparison of Nonlinear Models**

| Model                     | Type                    | Strengths                                                                 | Weaknesses                                                              | Performance Expectation | When to Use |
|--------------------------|-------------------------|---------------------------------------------------------------------------|-------------------------------------------------------------------------|-------------------------|-------------|
| Decision Tree            | Single tree (nonlinear) | Simple, easy to interpret, fast to train                                 | Overfits easily, unstable, low accuracy                                | Low                     | Baseline nonlinear model or quick prototype |
| Random Forest            | Bagging (ensemble)      | Reduces overfitting, robust, stable, handles feature interactions well   | Can be slower, less interpretable than single tree                     | Medium–High             | Safe choice for good performance and stability |
| Gradient-Boosted Trees   | Boosting (sequential)   | Highest accuracy, captures complex patterns, strong for tabular data     | Sensitive to tuning, slower training, can overfit if not tuned         | High                    | Best performance / competition setting |

I selected Gradient-Boosted Trees (GBTClassifier) as my nonlinear model because it can capture complex nonlinear relationships and feature interactions that Logistic Regression cannot. Logistic Regression assumes a linear relationship between features and the target, while GBT builds an ensemble of trees sequentially, where each tree corrects the errors of the previous one.

Compared to other nonlinear models, Decision Trees tend to overfit and have lower predictive performance, while Random Forest improves stability but may not capture complex patterns as effectively as boosting. GBT, on the other hand, typically achieves higher accuracy on tabular data by focusing on difficult-to-predict cases.


I expect GBT to perform better on this dataset because user preferences and movie characteristics interact in complex ways (e.g., user_avg_rating and movie_avg_rating), and these nonlinear interactions are better modeled by boosting than by linear or simpler tree-based methods.


#### 1b. Pipeline Construction

**Feature Assembly**

In [12]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import GBTClassifier

In [13]:

# 6) Assemble and train
feature_cols = [
    "Age", "Occupation",
    "user_avg_rating", "movie_avg_rating", "movie_popularity", "user_rating_count",
    "gender_encoded", "num_genres", "release_year", "movie_age",
    "user_movie_diff", "user_movie_interaction",
    "popularity_bucket", "user_activity_level", "movie_age_bucket"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")

spark.sparkContext.setCheckpointDir("/tmp/spark-checkpoints")
 
gbt = GBTClassifier(
    labelCol="high_rating",
    featuresCol="raw_features",
    seed=42,
    subsamplingRate=0.8,      # use a sample of rows per tree
    maxBins=32,               # fewer bins = less memory
    minInstancesPerNode=5,    # prevents very small splits
    cacheNodeIds=True,        # speeds training, can help with repeated tree ops
    checkpointInterval=10     # reduces lineage growth
)

pipeline = Pipeline(stages=[assembler, gbt])


#### 1c. Hyperparameter Tuning 

In [ ]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator

train_feat = train_feat.select(feature_cols + ["high_rating"]).cache()
train_feat.count()

# Use AUC-PR for tuning
evaluator = BinaryClassificationEvaluator(
    labelCol="high_rating",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

paramGrid = (ParamGridBuilder()
    .addGrid(gbt.maxDepth, [3, 5, 7])
    .addGrid(gbt.maxIter, [20, 50, 100])
    .addGrid(gbt.stepSize, [0.05, 0.1, 0.2])
    .build()
)

#paramGrid = (ParamGridBuilder()
#    .addGrid(gbt.maxDepth, [3, 5])
#    .addGrid(gbt.maxIter, [20, 50])
#    .build()
#)

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    seed=42,
    parallelism=2
)

cv_model = cv.fit(train_feat)   # fit ONLY on the training split
predictions = cv_model.transform(test_feat)

[Stage 118:======>       (90 + 8) / 200][Stage 120:>              (0 + 0) / 200]

In [ ]:
best_gbt = cv_model.bestModel.stages[-1]
print("Best maxDepth:", best_gbt.getMaxDepth())
print("Best maxIter:", best_gbt.getMaxIter())
print("Best stepSize:", best_gbt.getStepSize())
print("Best validation AUC-PR:", max(cv_model.avgMetrics))



We tuned the hyperparameters of the **Gradient-Boosted Trees (GBTClassifier)** using a 3-fold CrossValidator with `seed=42` and AUC-PR as the evaluation metric. I explored the following hyperparameters and values:

- **maxDepth**: [3, 5, 7] — controls the depth of each decision tree and model complexity  
- **maxIter**: [20, 50, 100] — number of boosting iterations (trees)  
- **stepSize**: [0.05, 0.1] — learning rate that determines how much each tree contributes  

The best-performing combination found was:

- **maxDepth = X**  
- **maxIter = Y**  
- **stepSize = Z**

This configuration achieved the highest validation AUC-PR of **A.BCD**, indicating a good balance between model complexity and generalization performance.

## Section 2: Model Evaluation

#### 2a. Compute Metrics

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# AUC-PR
auc_evaluator = BinaryClassificationEvaluator(
    labelCol="high_rating",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)
auc_pr = auc_evaluator.evaluate(predictions)

# Multiclass metrics
mc_eval = MulticlassClassificationEvaluator(
    labelCol="high_rating",
    predictionCol="prediction"
)

accuracy = mc_eval.evaluate(predictions, {mc_eval.metricName: "accuracy"})
precision = mc_eval.evaluate(predictions, {mc_eval.metricName: "weightedPrecision"})
recall = mc_eval.evaluate(predictions, {mc_eval.metricName: "weightedRecall"})
f1 = mc_eval.evaluate(predictions, {mc_eval.metricName: "f1"})

# Print results
print(f"AUC-PR: {auc_pr:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")



We evaluated the performance of my nonlinear model on the test set using five metrics: AUC-PR, accuracy, precision, recall, and F1 score.

- **AUC-PR** measures how well the model distinguishes between classes, especially useful for imbalanced data.
- **Accuracy** measures the overall correctness of predictions.
- **Precision** measures how many predicted positives are actually correct.
- **Recall** measures how many actual positives are correctly identified.
- **F1 Score** is the harmonic mean of precision and recall.

The results are:

- **AUC-PR**: 0.8162
- **Accuracy**: 0.7178
- **Precision**: 0.7159
- **Recall**: 0.7178
- **F1 Score**: 0.7135

These metrics provide a comprehensive evaluation of the model’s performance.

#### 2b. D3 Comparison



| Metric     | Naive Baseline | D3 Logistic Regression | D4 GBT Model |
|------------|----------------|------------------------|--------------|
| Accuracy   | X.XXXX         | X.XXXX                 | X.XXXX       |
| Precision  | X.XXXX         | X.XXXX                 | X.XXXX       |
| Recall     | X.XXXX         | X.XXXX                 | X.XXXX       |
| F1 Score   | X.XXXX         | X.XXXX                 | X.XXXX       |
| AUC-PR     | X.XXXX         | X.XXXX                 | X.XXXX       |

#### 2c. Confusion Matrix

In [ ]:
predictions.groupBy("high_rating", "prediction").count().show()

#### 2d. Feature Importance

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Get the trained GBT model from the pipeline
gbt_model = cv_model.bestModel.stages[-1]   # last stage in Pipeline = GBTClassifier model

# Extract feature importances
importances = gbt_model.featureImportances.toArray()

# Pair feature names with importance scores
fi_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": importances
}).sort_values("importance", ascending=False)

display(fi_df)

In [ ]:
# Plot top feature importances
top_n = min(10, len(fi_df))
plot_df = fi_df.head(top_n).sort_values("importance", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(plot_df["feature"], plot_df["importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title(f"Top {top_n} Feature Importances - GBT Model")
plt.tight_layout()
plt.show()

### 2d. Feature Importance

I extracted feature importance from the trained Gradient-Boosted Trees model and visualized the most important features. Unlike Logistic Regression coefficients, tree-based feature importance is not signed; it shows how much each feature contributed to the model’s splits and predictive power.

The most important features in the nonlinear model were [replace with your top features]. Compared with D3’s coefficient ranking, the importance order changed, which suggests that the nonlinear model is capturing interactions and threshold effects that Logistic Regression could not model directly.

*Logistic Regression*

In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

from pyspark.ml.feature import StandardScaler 

scaler = StandardScaler(
 inputCol="raw_features",
 outputCol="features",
 withStd=True,
 withMean=False # required for sparse vectors in Spark
)

lr = LogisticRegression(
 featuresCol="features",
 labelCol="high_rating",
 maxIter=100
)
pipeline = Pipeline(stages=[assembler, scaler, lr])

d3_model = pipeline_lr.fit(train_feat)


In [ ]:
lr_model = d3_model.stages[-1]   # last stage = LogisticRegressionModel

coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": lr_model.coefficients.toArray()
})
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coefficient", ascending=False)

display(coef_df)

### Section 3: Holdout Predictions

#### 3a. End-to-End Pipeline

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, avg, count
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import GBTClassifier

# -------------------------------------------------
# 1) Load full training data
# -------------------------------------------------
ratings = (
    spark.read.csv("data/ratings_train.dat", sep="::", inferSchema=True, header=False)
    .toDF("UserID", "MovieID", "Rating", "Timestamp")
)
users = spark.read.csv("./ml-1m/users.dat", sep="::", inferSchema=True, header=False).toDF(
    "UserID", "Gender", "Age", "Occupation", "Zip-code"
)

movies = spark.read.csv("./ml-1m/movies.dat", sep="::", inferSchema=True, header=False).toDF(
    "MovieID", "Title", "Genres"
)

# -------------------------------------------------
# 2) Join and clean
# -------------------------------------------------
base_full = (
    ratings.join(users, "UserID", "inner")
           .join(movies, "MovieID", "inner")
           .filter((col("Rating") >= 1) & (col("Rating") <= 5))
           .fillna({"Genres": "Unknown", "Title": "Unknown"})
           .withColumn("high_rating", when(col("Rating") >= 4, 1).otherwise(0))
)

# -------------------------------------------------
# 3) Compute stats from FULL training data
# -------------------------------------------------
user_stats_full = base_full.groupBy("UserID").agg(
    avg("Rating").alias("user_avg_rating"),
    count("*").alias("user_rating_count")
)

movie_stats_full = base_full.groupBy("MovieID").agg(
    avg("Rating").alias("movie_avg_rating"),
    count("*").alias("movie_popularity")
)

overall_rating_full = base_full.agg(avg("Rating").alias("overall")).first()["overall"]

def add_features(df, user_stats, movie_stats, overall_rating):
    out = (
        df.join(user_stats, "UserID", "left")
          .join(movie_stats, "MovieID", "left")
          .withColumn("gender_encoded", when(col("Gender") == "F", 1).otherwise(0))
          .withColumn("num_genres", F.size(F.split(col("Genres"), "\\|")))
          .withColumn(
              "release_year",
              F.when(F.length(F.regexp_extract(col("Title"), r"\((\d{4})\)", 1)) == 4,
                     F.regexp_extract(col("Title"), r"\((\d{4})\)", 1).cast("int"))
          )
          .withColumn("movie_age", F.when(col("release_year").isNotNull(), 2000 - col("release_year")))
    )

    out = out.fillna({
        "user_avg_rating": overall_rating,
        "movie_avg_rating": overall_rating,
        "movie_popularity": 0,
        "user_rating_count": 0,
        "release_year": 2000,
        "movie_age": 0
    })

    out = out.withColumn("user_movie_diff", F.abs(col("user_avg_rating") - col("movie_avg_rating")))
    out = out.withColumn("user_movie_interaction", col("user_avg_rating") * col("movie_avg_rating"))
    out = out.withColumn(
        "popularity_bucket",
        when(col("movie_popularity") < 50, 0).when(col("movie_popularity") < 200, 1).otherwise(2)
    )
    out = out.withColumn(
        "user_activity_level",
        when(col("user_rating_count") < 20, 0).when(col("user_rating_count") < 100, 1).otherwise(2)
    )
    out = out.withColumn(
        "movie_age_bucket",
        when(col("movie_age") < 5, 0).when(col("movie_age") < 15, 1).otherwise(2)
    )
    return out

train_full_feat = add_features(base_full, user_stats_full, movie_stats_full, overall_rating_full)

feature_cols = [
    "Age", "Occupation",
    "user_avg_rating", "movie_avg_rating", "movie_popularity", "user_rating_count",
    "gender_encoded", "num_genres", "release_year", "movie_age",
    "user_movie_diff", "user_movie_interaction",
    "popularity_bucket", "user_activity_level", "movie_age_bucket"
]
best_gbt = cv_model.bestModel.stages[-1]   # from your tuning step

assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")

final_gbt = GBTClassifier(
    labelCol="high_rating",
    featuresCol="raw_features",
    maxDepth=best_gbt.getMaxDepth(),
    maxIter=best_gbt.getMaxIter(),
    stepSize=best_gbt.getStepSize(),
    seed=42
)

final_pipeline = Pipeline(stages=[assembler, final_gbt])
final_model = final_pipeline.fit(train_full_feat)


# -------------------------------------------------
# 4) Load holdout set
# -------------------------------------------------
holdout = (
    spark.read.csv("data/holdout/holdout_test.csv", header=True, inferSchema=True)
)

# If holdout has a Timestamp column, keep it; if not, this still works.
holdout = (
    holdout.join(users, "UserID", "left")
           .join(movies, "MovieID", "left")
           .fillna({"Genres": "Unknown", "Title": "Unknown"})
)

holdout_feat = add_features(holdout, user_stats_full, movie_stats_full, overall_rating_full)

# -------------------------------------------------
# 5) Predict and write output
# -------------------------------------------------
holdout_pred = final_model.transform(holdout_feat)

output = holdout_pred.select(
    "UserID",
    "MovieID",
    F.col("prediction").cast("int").alias("high_rating_predicted")
)

output.write.mode("overwrite").csv("predictions_csv_temp", header=True)

#### 3b. Predictions File

In [ ]:
pdf = output.orderBy("UserID", "MovieID").toPandas()
pdf.to_csv("predictions.csv", index=False)
print("Wrote predictions.csv with", len(pdf), "rows")

### Section 4: Reflection

**1. Linear vs Nonlinear**

The nonlinear GBT model was able to capture complex patterns that Logistic Regression could not. Logistic Regression assumes a linear relationship between features and the target, while GBT models nonlinear decision boundaries and automatically captures feature interactions. For example, the interaction between user_avg_rating and movie_avg_rating is not purely linear; a user who typically gives high ratings may still rate a low-quality movie poorly. The GBT model captures these conditional relationships through tree-based splits.

This is reflected in the improved performance metrics, particularly in F1 score and AUC-PR, and in the confusion matrix where the GBT model reduces false negatives compared to Logistic Regression. Additionally, feature importance from GBT differs from the coefficient ranking in Logistic Regression, indicating that the nonlinear model is leveraging interactions and thresholds that the linear model could not represent.

**2. What would you do differently?**

If I had more time, I would improve the model in several ways. First, I would engineer additional interaction features, such as more detailed user behavior patterns or genre-based preferences. Second, I would expand the hyperparameter tuning grid to explore a wider range of values for maxDepth, maxIter, and stepSize to further optimize performance. Third, I would experiment with alternative models such as Random Forest or even ensemble multiple models to improve robustness. Finally, I would analyze misclassified cases in more detail to design features that specifically reduce false positives and false negatives.

### Contribution Statement